# Phase 5: Full Training — InfoNCE Loss + GCN + Random Walk Augmentation

**Producton training notebook.** Research-backed improvements over notebook 03:
- **InfoNCE loss** (cross-entropy over all candidates) instead of per-candidate BCE
- **Random walk data augmentation** (500K synthetic triples from the adjacency graph)
- **PageRank + HITS centrality features** added to node embeddings
- **GCN encoder** trained jointly with the scorer (not static, not broken)

Self-contained. Run from project root. Requires GPU for full training (~30 min).


In [ ]:
import io, pickle, tarfile, urllib.request, urllib.parse
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

ROOT = Path.cwd()
while not (ROOT / 'dataset-task2').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'dataset-task2'
CACHE = ROOT / '.cache'
CACHE.mkdir(exist_ok=True)
SUBMISSIONS = ROOT / 'submissions'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'ROOT: {ROOT}')
print(f'Device: {device}')
print(f'Cache: {CACHE}')


In [ ]:
articles = pd.read_csv(DATA / 'articles.csv')
categories = pd.read_csv(DATA / 'categories.csv')
train = pd.read_csv(DATA / 'states_train.csv')
test = pd.read_csv(DATA / 'states_test.csv')
sample_sub = pd.read_csv(DATA / 'sample_submission.csv')
print(f'Articles: {len(articles)}, Train: {len(train)}, Test: {len(test)}')
print(f'Categories: {len(categories)} rows, {categories["category"].nunique()} unique')


In [ ]:
def load_or_build_adjacency():
    cache_path = CACHE / 'wikispeedia_adj.pkl'
    data_path = DATA / 'wikispeedia_adj.pkl'
    for p in [cache_path, data_path]:
        if p.exists():
            with open(p, 'rb') as f:
                return pickle.load(f)
    import tarfile
    title_to_id = dict(zip(articles['title'].str.strip(), articles['article_id']))
    url = "https://snap.stanford.edu/data/wikispeedia/wikispeedia_paths-and-graph.tar.gz"
    print(f"Downloading Wikispeedia from {url} ...")
    with urllib.request.urlopen(url) as resp:
        with tarfile.open(fileobj=io.BytesIO(resp.read()), mode='r:gz') as tar:
            f = tar.extractfile("wikispeedia_paths-and-graph/links.tsv")
            content = f.read()
    links = pd.read_csv(io.BytesIO(content), sep='\t', skiprows=14,
                        header=None, names=['source', 'target'])
    links['source_decoded'] = links['source'].apply(lambda x: urllib.parse.unquote(x).replace('_', ' ').strip())
    links['target_decoded'] = links['target'].apply(lambda x: urllib.parse.unquote(x).replace('_', ' ').strip())
    links['source_id'] = links['source_decoded'].map(title_to_id)
    links['target_id'] = links['target_decoded'].map(title_to_id)
    links = links.dropna(subset=['source_id', 'target_id'])
    links['source_id'] = links['source_id'].astype(int)
    links['target_id'] = links['target_id'].astype(int)
    adj = {i: [] for i in range(4604)}
    for _, row in links.iterrows():
        adj[row['source_id']].append(row['target_id'])
    adj = {k: list(set(v)) for k, v in adj.items()}
    CACHE.mkdir(exist_ok=True)
    with open(cache_path, 'wb') as f:
        pickle.dump(adj, f)
    return adj

adj = load_or_build_adjacency()
n_edges = sum(len(v) for v in adj.values())
degrees = [len(v) for v in adj.values()]
print(f'Nodes: {len(adj)}, Edges: {n_edges}')
print(f'Degree: min={min(degrees)}, max={max(degrees)}, mean={np.mean(degrees):.1f}, median={np.median(degrees):.0f}')


## Feature Engineering

Load precomputed MiniLM embeddings + build category multi-hot + compute PageRank/HITS.

In [ ]:
embeddings = np.load(CACHE / 'article_embeddings.npy')
print(f'Embeddings: {embeddings.shape} {embeddings.dtype}')
assert embeddings.shape == (4604, 384)


In [ ]:
cat_set = sorted(categories['category'].unique())
cat_to_idx = {c: i for i, c in enumerate(cat_set)}
n_cats = len(cat_set)
print(f'Unique categories: {n_cats}')

cat_groups = categories.groupby('article_id')['category'].apply(list)
cat_enc = np.zeros((4604, n_cats), dtype=np.float32)
for aid, cats in cat_groups.items():
    for c in cats:
        cat_enc[aid, cat_to_idx[c]] = 1.0
print(f'Category encoding: {cat_enc.shape}')


### PageRank & HITS centrality scores

In [ ]:
def compute_pagerank(adj, n_nodes=4604, damping=0.85, n_iter=100):
    pr = np.ones(n_nodes, dtype=np.float64) / n_nodes
    adj_col = np.zeros((n_nodes, n_nodes), dtype=np.float64)
    for src, tgts in adj.items():
        if tgts:
            prob = 1.0 / len(tgts)
            for tgt in tgts:
                adj_col[src, tgt] = prob
    for _ in range(n_iter):
        pr = (1 - damping) / n_nodes + damping * (adj_col.T @ pr)
    return pr.astype(np.float32)

def compute_hits(adj, n_nodes=4604, n_iter=100):
    auth = np.ones(n_nodes, dtype=np.float64)
    hub = np.ones(n_nodes, dtype=np.float64)
    adj_bin = np.zeros((n_nodes, n_nodes), dtype=np.float64)
    for src, tgts in adj.items():
        for tgt in tgts:
            adj_bin[src, tgt] = 1.0
    for _ in range(n_iter):
        auth = adj_bin.T @ hub
        auth /= (auth.sum() + 1e-8)
        hub = adj_bin @ auth
        hub /= (hub.sum() + 1e-8)
    return hub.astype(np.float32), auth.astype(np.float32)

pr = compute_pagerank(adj)
hub, auth = compute_hits(adj)
print(f'PageRank: min={pr.min():.6f}, max={pr.max():.6f}, mean={pr.mean():.6f}')
print(f'Hub:      min={hub.min():.4f}, max={hub.max():.4f}')
print(f'Auth:     min={auth.min():.4f}, max={auth.max():.4f}')


### Combined node features

In [ ]:
node_feats = np.concatenate([
    embeddings,
    cat_enc,
    pr.reshape(-1, 1),
    hub.reshape(-1, 1),
    auth.reshape(-1, 1),
], axis=1).astype(np.float32)
feat_dim = node_feats.shape[1]
print(f'Node features: {node_feats.shape} (384 emb + {n_cats} cats + 3 centrality)')


## GCN Encoder

Two-layer GCN encoder trained jointly with the scorer. The GCN transforms raw node features through the link graph — neighbor features flow via message passing — producing graph-aware node embeddings used by the scoring MLP.

In [ ]:
def build_normalized_adj(adj, n_nodes):
    adj_np = np.zeros((n_nodes, n_nodes), dtype=np.float32)
    for s, ts in adj.items():
        for t in ts:
            adj_np[s, t] = 1.0
    deg = adj_np.sum(1, keepdims=True)
    deg[deg == 0] = 1
    d_inv_sqrt = np.power(deg, -0.5).flatten()
    adj_norm = adj_np * d_inv_sqrt[np.newaxis, :]
    adj_norm = adj_norm * d_inv_sqrt[:, np.newaxis]
    return adj_norm

adj_norm_np = build_normalized_adj(adj, 4604)
print(f'Normalized adjacency: {adj_norm_np.shape}, nonzeros: {(adj_norm_np > 0).sum()}')


In [ ]:
class GCNEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, out_dim=128):
        super().__init__()
        self.conv1 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.conv2 = nn.Linear(hidden_dim, out_dim, bias=False)

    def forward(self, x, adj_norm):
        x = F.relu(adj_norm @ self.conv1(x))
        x = adj_norm @ self.conv2(x)
        return x


## Random Walk Data Augmentation

Generate synthetic `(current, target, next)` triples by running random walks on the adjacency graph.
For each walk with length L, extract triples where the target is 2+ steps ahead of the current position.
Subsample to ~500K triples and cache to disk.
This is the Zaheer et al. (NeurIPS 2022) strategy — synthetic navigation data is within 10% of real.


In [ ]:
rng = np.random.default_rng(42)

def generate_synthetic_triples(adj, max_triples=500_000, max_walk_len=20, target_per_walk=5):
    nodes = list(adj.keys())
    node_weights = np.array([len(adj.get(n, [1])) for n in nodes], dtype=np.float64)
    node_weights /= node_weights.sum()

    triples = []
    pbar = tqdm(total=max_triples, desc='Generating walks')
    while len(triples) < max_triples:
        curr = int(nodes[rng.choice(len(nodes), p=node_weights)])
        path = [curr]
        for _ in range(max_walk_len - 1):
            neighbors = adj.get(curr, [])
            if not neighbors:
                break
            curr = int(rng.choice(neighbors))
            path.append(curr)
        L = len(path)
        if L < 3:
            continue
        n_sample = min(target_per_walk, (L - 1) * (L - 2) // 2)
        for _ in range(n_sample):
            i = int(rng.integers(0, L - 2))
            j = int(rng.integers(i + 2, L))
            triples.append((path[i], path[j], path[i + 1]))
            pbar.update(1)
            if len(triples) >= max_triples:
                break
    pbar.close()
    return np.array(triples, dtype=np.int32)

synth_cache = CACHE / 'synthetic_triples.npy'
if synth_cache.exists():
    synth_triples = np.load(synth_cache)
    print(f'Loaded {len(synth_triples)} synthetic triples from cache')
else:
    synth_triples = generate_synthetic_triples(adj, max_triples=500_000)
    np.save(synth_cache, synth_triples)
    print(f'Saved {len(synth_triples)} synthetic triples to {synth_cache}')

print(f'Synthetic triples: {synth_triples.shape}')
print(f'Sample: {synth_triples[:3]}')


## Datasets

`StateDataset` stores one entry per navigation state with all candidate outgoing links.
Variable-length candidates are padded to the max in each batch, with a mask to exclude padded positions from softmax.


In [ ]:
class StateDataset(Dataset):
    '''One sample = one state with all candidate outgoing links.'''
    def __init__(self, data, adj):
        self.items = []
        if isinstance(data, pd.DataFrame):
            source = data.itertuples(index=False)
            self._from_iter((r.current_article_id, r.target_article_id, r.next_article_id)
                           for r in source)
        else:
            self._from_iter(data)

    def _from_iter(self, rows):
        for curr, tgt, nxt in rows:
            links = adj.get(curr, [])
            if not links or nxt not in links:
                continue
            cands = list(links)
            self.items.append({
                'curr': curr,
                'tgt': tgt,
                'cands': cands,
                'deg': [len(adj.get(c, [])) for c in cands],
                'n_links': len(cands),
                'correct_idx': cands.index(nxt),
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        return (it['curr'], it['tgt'], it['cands'], it['deg'], it['n_links'], it['correct_idx'])


def collate_states(batch):
    B = len(batch)
    max_cands = max(len(b[2]) for b in batch)

    curr_ids = torch.tensor([b[0] for b in batch], dtype=torch.long)
    tgt_ids = torch.tensor([b[1] for b in batch], dtype=torch.long)
    cand_ids = torch.full((B, max_cands), -1, dtype=torch.long)
    cand_deg = torch.zeros(B, max_cands)
    mask = torch.zeros(B, max_cands, dtype=torch.bool)
    correct_idx = torch.tensor([b[5] for b in batch], dtype=torch.long)
    n_links = torch.tensor([b[4] for b in batch], dtype=torch.float)

    for i, b in enumerate(batch):
        n = len(b[2])
        cand_ids[i, :n] = torch.tensor(b[2], dtype=torch.long)
        cand_deg[i, :n] = torch.tensor(b[3], dtype=torch.float)
        mask[i, :n] = True

    return curr_ids, tgt_ids, cand_ids, cand_deg, n_links, correct_idx, mask


In [ ]:
real_ds = StateDataset(train, adj)
synth_ds = StateDataset(synth_triples, adj)
print(f'Real states: {len(real_ds)} (from {len(train)} train rows)')
print(f'Synthetic states: {len(synth_ds)} (from {len(synth_triples)} triples)')

from torch.utils.data import Subset

n_real = len(real_ds)
indices = np.random.default_rng(42).permutation(n_real)
split = int(n_real * 0.8)
train_indices = indices[:split]
val_indices = indices[split:]

train_ds = Subset(real_ds, train_indices.tolist())
val_dataset = Subset(real_ds, val_indices.tolist())
combined_ds = ConcatDataset([train_ds, synth_ds])
print(f'Train (real): {len(train_ds)}, Val (real): {len(val_dataset)}')
print(f'Combined train: {len(combined_ds)} (real + synthetic)')

train_loader = DataLoader(combined_ds, batch_size=32, shuffle=True, collate_fn=collate_states,
                          num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_states,
                        num_workers=0, pin_memory=True)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')


## Model: GraphConditionedScorer

1. **GCN Encoder** runs once per batch (neutral labels), producing [4604, 128] graph-aware node embeddings.
2. **Scoring MLP** takes `[gcn_curr ‖ gcn_tgt ‖ gcn_cand ‖ feat_curr ‖ feat_tgt ‖ feat_cand ‖ out_deg ‖ n_links]` per candidate and outputs logits.
3. **InfoNCE loss** = cross-entropy over all candidates of a state.


In [ ]:
class GraphConditionedScorer(nn.Module):
    def __init__(self, node_feats_np, adj_norm_np, gcn_hidden=256, gcn_out=128, scorer_hidden=256):
        super().__init__()
        self.register_buffer('node_feats', torch.from_numpy(node_feats_np))
        self.register_buffer('adj_norm', torch.from_numpy(adj_norm_np))
        gcn_in = node_feats_np.shape[1]
        self.gcn = GCNEncoder(gcn_in, gcn_hidden, gcn_out)
        scorer_in = gcn_out * 3 + gcn_in * 3 + 2
        self.scorer = nn.Sequential(
            nn.Linear(scorer_in, scorer_hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(scorer_hidden, scorer_hidden // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(scorer_hidden // 2, 1),
        )

    def forward(self, curr_ids, tgt_ids, cand_ids, cand_deg, n_links, mask):
        B, N = cand_ids.shape

        node_feats = self.node_feats
        adj_norm = self.adj_norm

        node_embs = self.gcn(node_feats, adj_norm)

        gcn_curr = node_embs[curr_ids].unsqueeze(1).expand(-1, N, -1)
        gcn_tgt = node_embs[tgt_ids].unsqueeze(1).expand(-1, N, -1)
        gcn_cand = node_embs[cand_ids.clamp(min=0)]

        feat_curr = node_feats[curr_ids].unsqueeze(1).expand(-1, N, -1)
        feat_tgt = node_feats[tgt_ids].unsqueeze(1).expand(-1, N, -1)
        feat_cand = node_feats[cand_ids.clamp(min=0)]

        x = torch.cat([
            gcn_curr, gcn_tgt, gcn_cand,
            feat_curr, feat_tgt, feat_cand,
            cand_deg.unsqueeze(-1),
            n_links.unsqueeze(-1).unsqueeze(-1).expand(-1, N, -1),
        ], dim=-1)

        scores = self.scorer(x).squeeze(-1)
        scores = scores.masked_fill(~mask, float('-inf'))
        return scores


model = GraphConditionedScorer(node_feats, adj_norm_np).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')


## Training

- **Loss**: InfoNCE (cross-entropy) — scores all candidates per state, softmax, maximizes correct link probability.
- **Optimizer**: AdamW (lr=1e-3, weight decay=1e-5) + CosineAnnealingLR
- **Epochs**: 30 (early stopping at best val accuracy)


In [ ]:
from time import perf_counter

def train_epoch(model, loader, opt):
    model.train()
    total_loss = 0
    for batch in loader:
        curr_ids, tgt_ids, cand_ids, cand_deg, n_links, correct_idx, mask = \
            [x.to(device) for x in batch]

        scores = model(curr_ids, tgt_ids, cand_ids, cand_deg, n_links, mask)
        loss = F.cross_entropy(scores, correct_idx)

        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for batch in loader:
        curr_ids, tgt_ids, cand_ids, cand_deg, n_links, correct_idx, mask = \
            [x.to(device) for x in batch]

        scores = model(curr_ids, tgt_ids, cand_ids, cand_deg, n_links, mask)
        pred_idx = scores.argmax(dim=-1)
        correct += (pred_idx == correct_idx).sum().item()
        total += len(correct_idx)
    return correct / total


epochs = 30
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

best_val_acc = 0.0
history = []

print(f'{"Epoch":>6} | {"Loss":>8} | {"Val Acc":>8} | {"Time":>8} | {"LR":>8}')
print('-' * 60)

for ep in range(epochs):
    t0 = perf_counter()
    loss = train_epoch(model, train_loader, opt)
    sched.step()
    val_acc = evaluate(model, val_loader)
    elapsed = perf_counter() - t0

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    history.append((loss, val_acc))
    print(f'  {ep+1:3d}  | {loss:8.4f} | {val_acc*100:7.2f}% | {elapsed:7.1f}s | {sched.get_last_lr()[0]:.6e}')

model.load_state_dict(best_state)
print(f'\nBest val accuracy: {best_val_acc*100:.2f}%')


## Test Prediction

Score all candidate links for each test state. Fallback: if current article has no outgoing links, predict the target article.


In [ ]:
@torch.no_grad()
def predict_test(model, test_df, adj, node_feats_np, adj_norm_np):
    model.eval()
    preds = []
    for _, r in tqdm(test_df.iterrows(), total=len(test_df), desc='Predict'):
        curr, tgt = r['current_article_id'], r['target_article_id']
        links = adj.get(curr, [])
        if not links:
            preds.append(tgt)
            continue

        cands = list(links)
        B = len(cands)

        curr_ids = torch.tensor([curr], dtype=torch.long, device=device)
        tgt_ids = torch.tensor([tgt], dtype=torch.long, device=device)
        cand_ids = torch.tensor([cands], dtype=torch.long, device=device)
        cand_deg = torch.tensor([[len(adj.get(c, [])) for c in cands]], dtype=torch.float, device=device)
        n_links = torch.tensor([B], dtype=torch.float, device=device)
        mask = torch.ones(1, B, dtype=torch.bool, device=device)

        scores = model(curr_ids, tgt_ids, cand_ids, cand_deg, n_links, mask)
        preds.append(cands[scores[0].argmax().item()])

    return np.array(preds)

test_preds = predict_test(model, test, adj, node_feats, adj_norm_np)
print(f'Predicted {len(test_preds)} test states')


## Submission

Save prediction CSV to `submissions/{timestamp}_dl_infonce/` and validate against the sample submission format.


In [ ]:
def make_submission(state_ids, predictions, path):
    sub = pd.DataFrame({'state_id': state_ids, 'predicted_next_article_id': predictions})
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(path, index=False)
    return sub

def validate_submission(path):
    expected = sample_sub
    actual = pd.read_csv(path)
    assert list(actual.columns) == list(expected.columns), f'Columns: {list(actual.columns)}'
    assert len(actual) == len(expected), f'Rows: {len(actual)} != {len(expected)}'
    assert list(actual['state_id']) == list(expected['state_id']), 'state_id mismatch'
    return True

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
sub_dir = SUBMISSIONS / f'{ts}_dl_infonce'
sub_dir.mkdir(parents=True, exist_ok=True)
sub_path = sub_dir / 'submission.csv'
make_submission(test['state_id'].values, test_preds, sub_path)
validate_submission(sub_path)
print(f'\nSubmission saved: {sub_path}')


## Summary

In [ ]:
n_real_states = len(real_ds)
n_synth_states = len(synth_ds)
n_total_train = len(combined_ds)
n_params = sum(p.numel() for p in model.parameters())

print(f'Model: GraphConditionedScorer ({n_params:,} params)')
print(f'  GCN: {feat_dim}d → 256d → 128d (2-layer)')
print(f'  Scorer: 3-layer MLP ({model.scorer[0].in_features}d → 1)')
print(f'')
print(f'Training data:')
print(f'  Real states:      {n_real_states:>8,}')
print(f'  Synthetic states: {n_synth_states:>8,}')
print(f'  Total:            {n_total_train:>8,}')
print(f'  Val states:       {len(val_dataset):>8,}')
print(f'')
print(f'Results:')
print(f'  Best val accuracy: {best_val_acc*100:.2f}%')
print(f'  Epochs trained:    {epochs}')
print(f'')
print(f'Baselines (from notebook 02):')
print(f'  Most popular link:  62.5%')
print(f'  XGBoost:            50.0%')
print(f'  Title similarity:   16.5%')
print(f'  Category overlap:   12.2%')
print(f'')
print(f'Features:')
print(f'  Text embeddings:   {embeddings.shape[1]}d (MiniLM)')
print(f'  Category encoding: {cat_enc.shape[1]}d (multi-hot)')
print(f'  Centrality:        3d (PageRank + Hub + Authority)')
print(f'  Total raw:         {feat_dim}d')
print(f'')
print(f'Submissions directory: {sub_dir}')
